In [1]:
import sqlite3
import pandas as pd

In [2]:
df = pd.read_csv("dataset/customer_churn.csv")

In [3]:
conn = sqlite3.connect("customer_churn.db")

In [4]:
df.to_sql(
    "customer_churn",
    conn,
    if_exists="replace",
    index=False
)

5000

In [5]:
pd.read_sql_query(
    "SELECT * FROM customer_churn LIMIT 5;",
    conn
)

,Customer ID,Gender,Age,Tenure Months,Contract,Internet Service,Payment Method,Tech Support,Paperless Billing,Monthly Charges,Churn,Annual Revenue,Customer Segment
0,CUST-00001,Male,28,25,Month-to-month,DSL,Mailed check,No,Yes,105.61,No,1267.32,25-48 Months
1,CUST-00002,Female,19,16,One year,Fiber optic,Credit card,Yes,Yes,111.88,No,1342.56,13-24 Months
2,CUST-00003,Female,51,38,Month-to-month,Fiber optic,Electronic check,Yes,Yes,102.42,Yes,1229.04,25-48 Months
3,CUST-00004,Male,54,48,Month-to-month,No internet,Bank transfer,No,No,67.99,Yes,815.88,25-48 Months
4,CUST-00005,Male,58,19,Month-to-month,DSL,Mailed check,No,Yes,32.06,Yes,384.72,13-24 Months


In [6]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customer_churn;
"""

result = pd.read_sql_query(query, conn)

result

,total_customers,churned_customers,churn_rate
0,5000,1330,26.6


In [7]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customer_churn
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,2723,950,34.89
1,One year,1295,262,20.23
2,Two year,982,118,12.02


In [8]:
query = """
SELECT
    "Payment Method",
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customer_churn
GROUP BY "Payment Method"
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,Payment Method,total_customers,churned_customers,churn_rate
0,Electronic check,1677,563,33.57
1,Credit card,1105,260,23.53
2,Bank transfer,1159,266,22.95
3,Mailed check,1059,241,22.76


In [9]:
query = """
SELECT
    "Internet Service",
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customer_churn
GROUP BY "Internet Service"
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,Internet Service,total_customers,churned_customers,churn_rate
0,Fiber optic,2227,709,31.84
1,DSL,2094,473,22.59
2,No internet,679,148,21.80


In [10]:
query = """
SELECT
    CASE
        WHEN "Tenure Months" <= 12 THEN '0-12 Months'
        WHEN "Tenure Months" <= 24 THEN '13-24 Months'
        WHEN "Tenure Months" <= 36 THEN '25-36 Months'
        WHEN "Tenure Months" <= 48 THEN '37-48 Months'
        WHEN "Tenure Months" <= 60 THEN '49-60 Months'
        ELSE '60+ Months'
    END AS tenure_group,

    COUNT(*) AS total_customers,

    SUM(
        CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END
    ) AS churned_customers,

    ROUND(
        100.0 *
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate

FROM customer_churn

GROUP BY tenure_group

ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,tenure_group,total_customers,churned_customers,churn_rate
0,0-12 Months,860,262,30.47
1,13-24 Months,811,224,27.62
2,25-36 Months,834,223,26.74
3,60+ Months,817,210,25.70
4,37-48 Months,882,219,24.83
5,49-60 Months,796,192,24.12


In [11]:
query = """
SELECT
    Churn,
    ROUND(AVG("Monthly Charges"), 2) AS average_monthly_charges
FROM customer_churn
GROUP BY Churn;
"""

pd.read_sql_query(query, conn)

,Churn,average_monthly_charges
0,No,72.36
1,Yes,72.47


## SQL Analysis Summary

- Total customers: 5,000
- Churned customers: 1,330
- Overall churn rate: 26.6%
- Month-to-month contracts have the highest churn rate at approximately 34.9%.
- Electronic check users have the highest churn rate at approximately 33.6%.
- Fiber optic customers have the highest churn rate at approximately 31.84%.
- Customers with 0-12 months of tenure have the highest churn rate at approximately 30.47%.
- Average monthly charges are very similar between churned and retained customers.